# RF15 — Modo Tentativa

CRUD completo de tentativas de resolução, com dashboard de tentativas por status e por pergunta.

Antes de rodar, crie um .env na raiz do projeto com:
DB_HOST=localhost
DB_NAME=exercicios_colaborativos
DB_USER=postgres
DB_PASSWORD=sua_senha

In [1]:
%pip install pandas sqlalchemy psycopg2-binary panel python-dotenv jupyter_bokeh matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ===================================================
# RF15 — Modo Tentativa
# Autor: Ivna Leite
# ===================================================

import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn
import matplotlib.pyplot as plt

In [3]:
# Carrega as variaveis do arquivo .env
load_dotenv()

True

In [4]:
# Le as variaveis de ambiente
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

In [5]:
# Cria conexao com psycopg2 usando as variaveis carregadas
con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)

In [6]:
# Define a string de conexao para o SQLAlchemy, utilizando as variaveis do .env
# Cria o objeto engine do SQLAlchemy que sera usado para conectar e executar comandos no banco

cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

sqlalchemy.create_engine(cnx)

engine = create_engine(cnx)

In [7]:
# Executa a consulta SQL para buscar todos os
# registros da tabela 'Tentativa' no banco PostgreSQL
# e carrega o resultado em um DataFrame do pandas

query = "select * from Tentativa;"
df = pd.read_sql_query(query, cnx)

df

,id_tentativa,texto_resolucao,data_hora,status_correto,id_estudante,id_pergunta
0,1,Tentei usar substituição simples mas não funci...,2026-06-10 13:10:56.018428,False,1,1
1,2,Vetor tem direção e escalar não.,2026-06-10 13:10:56.018428,True,2,2
2,3,Adicionei H2O para balancear mas errei os coef...,2026-06-10 13:10:56.018428,False,3,3
3,4,Fiz a função chamar ela mesma sem caso base e ...,2026-06-10 13:10:56.018428,False,6,4
4,5,Coloquei a FK na tabela errada.,2026-06-10 13:10:56.018428,False,7,5
5,6,Achei que era -sen(x).,2026-06-10 13:10:56.018428,False,1,6
6,7,Calculei só o determinante 2x2.,2026-06-10 13:10:56.018428,False,9,7
7,8,Somei todos e dividi pela quantidade.,2026-06-10 13:10:56.018428,False,2,8
8,9,Lembrei só de 5 camadas do OSI.,2026-06-10 13:10:56.018428,False,3,9
9,10,Desenhei o diagrama sem chaves primárias.,2026-06-10 13:10:56.018428,False,6,10


In [8]:
# Inicializa as extensoes do Panel necessarias para exibir tabelas
# interativas (Tabulator) e notificacoes na interface grafica

pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)
pn.extension('matplotlib')

In [9]:
# campos de texto

# declare esta variavel para usar na consulta de campos em branco
flag = ''

# Cria widgets interativos para o usuario inserir ou selecionar dados.
# Os dropdowns de estudante e pergunta sao preenchidos a partir do banco,
# evitando que se digite um id_estudante ou id_pergunta inexistente.

def carregar_opcoes_estudante():
    df = pd.read_sql_query("""
        SELECT e.id_usuario, u.p_nome, u.sobrenome
        FROM Estudante e JOIN Usuario u ON e.id_usuario = u.id_usuario
        ORDER BY e.id_usuario
    """, cnx)
    opcoes = {"— Selecione —": None}
    for _, r in df.iterrows():
        opcoes[f"{r['p_nome']} {r['sobrenome']} (#{int(r['id_usuario'])})"] = int(r['id_usuario'])
    return opcoes

def carregar_opcoes_pergunta():
    df = pd.read_sql_query("""
        SELECT id_conteudo, titulo_resumido FROM Pergunta ORDER BY id_conteudo
    """, cnx)
    opcoes = {"— Selecione —": None}
    for _, r in df.iterrows():
        opcoes[f"#{int(r['id_conteudo'])} - {r['titulo_resumido']}"] = int(r['id_conteudo'])
    return opcoes

estudante_tentativa = pn.widgets.Select(
    name="Estudante *",
    options=carregar_opcoes_estudante()
)

pergunta_tentativa = pn.widgets.Select(
    name="Pergunta *",
    options=carregar_opcoes_pergunta()
)

texto_resolucao = pn.widgets.TextAreaInput(
    name="Tentativa de Resolucao",
    value='',
    placeholder='Digite o que o estudante tentou responder',
    disabled=False
)

status_correto = pn.widgets.Checkbox(name='Marcar como Correta', value=False)

filtro_busca = pn.widgets.TextInput(
    name="Buscar (trecho do texto da tentativa)",
    value='',
    placeholder='Digite para filtrar...',
    disabled=False
)

id_tentativa_alvo = pn.widgets.IntInput(
    name="ID da Tentativa (para Atualizar/Excluir)",
    value=0
)

In [10]:
# Cria quatro botoes para as acoes principais da aplicacao CRUD:
# Consultar, Inserir, Excluir e Atualizar registros no banco de dados

buttonConsultar = pn.widgets.Button(name='Consultar', button_type='default')

buttonInserir = pn.widgets.Button(name='Inserir', button_type='default')

buttonExcluir = pn.widgets.Button(name='Excluir', button_type='default')

buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='default')

In [11]:
def queryAll():
    """
    Consulta todos os registros da tabela 'Tentativa' no banco de dados, com
    o nome do estudante e o titulo da pergunta relacionada, e retorna um
    widget Tabulator para exibicao interativa dos dados.

    Returns:
        pn.widgets.Tabulator: Widget que exibe a tabela com todos os dados da tabela 'Tentativa'.
    """
    query = """
        SELECT t.id_tentativa, u.p_nome || ' ' || u.sobrenome AS estudante,
               p.titulo_resumido AS pergunta, t.texto_resolucao,
               t.status_correto, t.data_hora
        FROM Tentativa t
        JOIN Usuario u ON t.id_estudante = u.id_usuario
        JOIN Pergunta p ON t.id_pergunta = p.id_conteudo
        ORDER BY t.data_hora DESC
    """
    df = pd.read_sql_query(query, cnx)
    return pn.widgets.Tabulator(df)


def limpar_campos():
    """
    Limpa todos os campos do formulario, deixando-os prontos para uma nova operacao.
    """
    estudante_tentativa.value = None
    pergunta_tentativa.value = None
    texto_resolucao.value = ''
    status_correto.value = False
    id_tentativa_alvo.value = 0


# consultar
# neste exemplo o metodo de consulta usa o dataframe do pandas como retorno. Note que a flag e usada para ignorar quando um
# campo for null (condicao e sempre verdadeira).
def on_consultar():
    """
    Consulta registros na tabela 'Tentativa' filtrando por um trecho do texto
    de resolucao informado. Se o campo estiver vazio, retorna todos os registros.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela com os dados encontrados ou alerta em caso de erro.
    """
    try:
        query = f"""
            SELECT t.id_tentativa, u.p_nome || ' ' || u.sobrenome AS estudante,
                   p.titulo_resumido AS pergunta, t.texto_resolucao,
                   t.status_correto, t.data_hora
            FROM Tentativa t
            JOIN Usuario u ON t.id_estudante = u.id_usuario
            JOIN Pergunta p ON t.id_pergunta = p.id_conteudo
            WHERE ('{filtro_busca.value_input}'='{flag}' OR t.texto_resolucao ILIKE '%{filtro_busca.value_input}%')
            ORDER BY t.data_hora DESC
        """
        df = pd.read_sql_query(query, cnx)
        table = pn.widgets.Tabulator(df)
        return table
    except:
        return pn.pane.Alert('Não foi possível consultar!')


def on_inserir():
    """
    Insere uma nova tentativa na tabela 'Tentativa', vinculada a um estudante
    e a uma pergunta existentes. Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        if not estudante_tentativa.value or not pergunta_tentativa.value:
            return pn.pane.Alert('Selecione o estudante e a pergunta.')

        cursor = con.cursor()
        cursor.execute(
            "insert into Tentativa(texto_resolucao, status_correto, id_estudante, id_pergunta) VALUES (%s, %s, %s, %s)",
            (texto_resolucao.value_input, status_correto.value, estudante_tentativa.value, pergunta_tentativa.value)
        )
        cursor.query
        con.commit()
        pn.state.notifications.success('Tentativa inserida com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível inserir: {str(e)}')


def on_atualizar():
    """
    Atualiza o texto de resolucao e o status (correto/incorreto) da tentativa
    identificada pelo ID informado em id_tentativa_alvo.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor = con.cursor()
        cursor.execute(
            "UPDATE Tentativa SET texto_resolucao = %s, status_correto = %s WHERE id_tentativa = %s",
            (texto_resolucao.value_input, status_correto.value, id_tentativa_alvo.value)
        )
        cursor.query
        con.commit()
        pn.state.notifications.success('Tentativa atualizada com sucesso!')
        limpar_campos()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível atualizar: {str(e)}')


def on_excluir():
    """
    Exclui o registro da tabela 'Tentativa' com o ID informado em id_tentativa_alvo.
    Ao final, limpa o formulario automaticamente.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor = con.cursor()
        cursor.execute("delete from Tentativa where id_tentativa=%s", (id_tentativa_alvo.value,))
        rows_deleted = cursor.rowcount
        con.commit()
        pn.state.notifications.success('Tentativa excluida com sucesso!')
        limpar_campos()
        return queryAll()
    except:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert('Não foi possível excluir!')

In [12]:
# Funcao que chama a acao correta (consultar, inserir, atualizar ou excluir)
# dependendo do botao que foi clicado (representado pelos parametros booleanos)

def table_creator(cons, ins, atu, exc):
    if cons:
        return on_consultar()
    if ins:
        return on_inserir()
    if atu:
        return on_atualizar()
    if exc:
        return on_excluir()

In [13]:
# Cria uma ligacao interativa (bind) entre os botoes e a funcao que executa a acao correspondente,
# atualizando a tabela na interface sempre que algum botao for clicado.

interactive_table = pn.bind(table_creator, buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir)

In [14]:
def criar_graficos_pandas(event=None):
    query_status = """
        SELECT CASE WHEN status_correto THEN 'Correta' ELSE 'Incorreta' END AS status,
               COUNT(*) as qtd
        FROM Tentativa
        GROUP BY status_correto
    """
    query_pergunta = """
        SELECT p.titulo_resumido AS pergunta, COUNT(*) as qtd
        FROM Tentativa t
        JOIN Pergunta p ON t.id_pergunta = p.id_conteudo
        GROUP BY p.titulo_resumido
        ORDER BY qtd DESC
        LIMIT 10
    """

    def_status = pd.read_sql(query_status, engine)

    def_pergunta = pd.read_sql(query_pergunta, engine)

    fig1, ax1 = plt.subplots(figsize=(6, 5))

    fig2, ax2 = plt.subplots(figsize=(6, 5))

    if not def_status.empty:
        def_status.set_index('status').plot(
            kind='pie',
            y='qtd',
            ax=ax1,
            autopct='%1.1f%%',
            legend=False,
            title='Tentativas por Status',
            ylabel=''
        )
    else:
        ax1.text(0.5, 0.5, "Sem dados", ha='center')

    if not def_pergunta.empty:
        def_pergunta.set_index('pergunta').plot(
            kind='barh',
            y='qtd',
            ax=ax2,
            color='#4CAF50',
            legend=False,
            title='Tentativas por Pergunta (Top 10)'
        )
        ax2.set_xlabel('Quantidade de Tentativas')
        ax2.invert_yaxis()
    else:
        ax2.text(0.5, 0.5, "Sem dados", ha='center')

    plt.close(fig1)
    plt.close(fig2)

    return pn.Column(
        pn.pane.Matplotlib(fig1, tight=True),
        pn.pane.Matplotlib(fig2, tight=True),
        name="Graficos"
    )

In [15]:
# Monta o layout da interface com Panel:
# - Coluna esquerda com o titulo, os campos de entrada e os botoes de acao
# - Coluna direita com a tabela interativa que mostra os dados do banco
# O metodo `.servable()` permite que essa interface seja exibida ao rodar o Panel server

layout_crud = pn.Row(
    pn.Column(
        'Tentativa CRUD',
        estudante_tentativa, pergunta_tentativa, texto_resolucao, status_correto, filtro_busca, id_tentativa_alvo,
        pn.Row(buttonConsultar, buttonInserir),
        pn.Row(buttonAtualizar, buttonExcluir),
    ),
    pn.Column(interactive_table),
    name="Gerenciamento"
)

layout_dashboard = pn.bind(criar_graficos_pandas, buttonInserir)

aba_principal = pn.Tabs(
    layout_crud,
    ("Dashboard (Pandas)", layout_dashboard)
)

aba_principal.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'f85b052e-1c44-4776-bd34-20da6b9ad5f6': {'version…